# Phase 4: Phylogenetic Filtering (Resolving Galton's Problem)

**Date:** 16 avril 2026  
**Objective:** Apply phylogenetic filtering to D-PLACE data to resolve Galton's problem (phylogenetic non-independence).

## Overview

Galton's Problem: Cross-cultural data points are not independent—cultures share evolutionary history through language descent. 

**Solution:** Two-pronged robustness approach:
1. **Primary Analysis:** Filter to one culture per language family (~150–200 independent cultures)
2. **Robustness Check:** Full dataset (2,087 cultures) for sensitivity comparison

**Reference:** Mace & Pagel (1994). "The comparative method in anthropology." *Current Anthropology* 35(3): 87–92.

---

This notebook generates the phylogenetically-filtered dataset for Phase 4 clustering.

## Section 1: Import Required Libraries & Load Harmonised Data

In [ ]:
# Phase 4: Imports & Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from analysis.phylogenetic import (
    filter_one_per_language_family,
    compute_phylogenetic_summary,
    create_robustness_dataset_pair
)
from analysis.config import CLUSTERING_FEATURES

# Configuration
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'harmonised'
OUTPUTS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'clusters'
VIZ_PATH = PROJECT_ROOT / 'data' / 'visualizations' / 'clusters'

# Create output directories
OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)
VIZ_PATH.mkdir(parents=True, exist_ok=True)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports complete")
print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Output paths created")

In [ ]:
# Load harmonised D-PLACE data
dplace_harmonised_path = DATA_PATH / 'dplace_harmonised.parquet'

print(f"Loading: {dplace_harmonised_path}")
dplace_df = pd.read_parquet(dplace_harmonised_path)

print(f"\n✓ Loaded D-PLACE harmonised data")
print(f"  Shape: {dplace_df.shape}")
print(f"  Columns: {list(dplace_df.columns[:10])}... ({len(dplace_df.columns)} total)")
print(f"\nData types:")
print(dplace_df.dtypes.value_counts())
print(f"\nFirst few rows:")
dplace_df.head()

## Section 2: Verify Schema & Language Family Column

In [ ]:
# Verify harmonised schema - 22 harmonised features expected
expected_features = [
    'trance_state', 'possession_spirit', 'soul_journey', 'entheogen_use',
    'specialist_type', 'initiatory_crisis', 'hereditary_role',
    'layered_cosmology', 'animal_transformation', 'ancestor_mediation', 'nature_spirits',
    'rhythmic_percussion', 'healing_function', 'divination', 'public_performance', 'chanting_singing',
    # Plus any composite indicators from Phase 3
]

# Check for key required columns
required_cols = ['culture_id', 'culture_name', 'language_family', 'lat', 'lon', 'time_start_ce', 'time_end_ce']

print("Schema Validation:")
print("=" * 60)
for col in required_cols:
    if col in dplace_df.columns:
        print(f"✓ {col:25s} present")
    else:
        print(f"✗ {col:25s} MISSING!")

print(f"\nLanguage family column integrity:")
print(f"  Non-null values: {dplace_df['language_family'].notna().sum():,} / {len(dplace_df):,}")
print(f"  Unique language families: {dplace_df['language_family'].nunique():,}")
print(f"  Missing: {dplace_df['language_family'].isna().sum()}")

if dplace_df['language_family'].isna().sum() > 0:
    print("\n⚠ WARNING: Language family has missing values!")
    print(f"  Action: Filtering operations will exclude NA records")

print(f"\nTop 10 most represented language families:")
top_families = dplace_df['language_family'].value_counts().head(10)
for family, count in top_families.items():
    print(f"  {family:30s}: {count:4d} cultures")

## Section 3: Phylogenetic Filtering - Galton's Problem Resolution

**Approach:** Two-pronged robustness framework
- **Primary:** Filter to one representative culture per language family (~150–200 cultures)
- **Robustness:** Full dataset for comparison (2,087 cultures)

In [ ]:
# Generate paired datasets for two-pronged analysis
print("PHYLOGENETIC FILTERING")
print("=" * 70)
print("\nStep 1: Apply one-culture-per-language-family filter")
print(f"  Input: {len(dplace_df):,} cultures in {dplace_df['language_family'].nunique():,} language families")

# Filter to one per language family
dplace_filtered = filter_one_per_language_family(
    dplace_df,
    language_family_col='language_family',
    random_state=42
)

print(f"  Output (filtered): {len(dplace_filtered):,} cultures")
print(f"  Output (full, for robustness): {len(dplace_df):,} cultures")
print(f"\n  Reduction ratio: {len(dplace_filtered)/len(dplace_df):.1%}")

# Mark dataset types
dplace_filtered['dataset_type'] = 'phylo_filtered'
dplace_full = dplace_df.copy()
dplace_full['dataset_type'] = 'full_dataset'

print("\n✓ Phylogenetic filtering complete")

In [ ]:
# Detailed analysis of language family distribution
print("\nLanguage Family Distribution Analysis:")
print("=" * 70)

# Family sizes before filtering
family_sizes = dplace_df['language_family'].value_counts()
print(f"Distribution before filtering:")
print(f"  Min cultures per family: {family_sizes.min()}")
print(f"  Max cultures per family: {family_sizes.max()}")
print(f"  Mean cultures per family: {family_sizes.mean():.1f}")
print(f"  Median cultures per family: {family_sizes.median():.0f}")

# Family sizes after filtering (should be all 1)
filtered_family_sizes = dplace_filtered['language_family'].value_counts()
print(f"\nDistribution after filtering (phylo-independent):")
print(f"  All families have 1 culture: {(filtered_family_sizes == 1).all()}")
print(f"  Number of families: {len(filtered_family_sizes)}")

# Show families with high redundancy (affected most by filtering)
high_redundancy = family_sizes[family_sizes > 10].sort_values(ascending=False).head(10)
print(f"\nTop 10 most represented families (before filtering):")
for family, count in high_redundancy.items():
    reduction = count - 1
    print(f"  {family:30s}: {count:3d} → 1 culture (removed {reduction:2d})")

In [ ]:
# Visualise language family redundancy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribution of cultures per language family
ax1 = axes[0]
family_sizes_all = dplace_df['language_family'].value_counts().values
ax1.hist(family_sizes_all, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(family_sizes_all.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {family_sizes_all.mean():.1f}')
ax1.axvline(1, color='green', linestyle='--', linewidth=2, label='After filtering')
ax1.set_xlabel('Number of Cultures per Language Family')
ax1.set_ylabel('Frequency')
ax1.set_title('Language Family Redundancy (Before Filtering)\nGalton\'s Problem: Phylogenetic Non-Independence')
ax1.legend()
ax1.set_xlim(0, min(100, family_sizes_all.max() + 10))

# Plot 2: Cumulative reduction from filtering
ax2 = axes[1]
cumulative_removed = np.cumsum([count - 1 for count in family_sizes_all])
cumulative_kept = len(dplace_df) - cumulative_removed
x_labels = np.arange(len(cumulative_kept)) + 1

# Show only subset for readability
subset_size = min(200, len(cumulative_kept))
ax2.plot(range(subset_size), cumulative_kept[:subset_size], color='steelblue', linewidth=2, label='Cultures kept after filtering')
ax2.fill_between(range(subset_size), cumulative_kept[:subset_size], len(dplace_df), alpha=0.3, color='red', label='Cultures removed (phylo redundancy)')
ax2.set_xlabel('Language Family Rank')
ax2.set_ylabel('Cumulative Count')
ax2.set_title('Cumulative Impact of Phylogenetic Filtering\n(First 200 language families)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(VIZ_PATH / 'phylogenetic_filtering_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Visualisation saved: phylogenetic_filtering_analysis.png")

## Section 4: Geographic & Temporal Representation
What cultures are selected for primary analysis vs. excluded for phylogenetic redundancy?

In [ ]:
# Compare geographic distributions
print("\nGEOGRAPHIC REPRESENTATION:")
print("=" * 70)

# Calculate rough geographic regions
def assign_region(lat):
    """Assign broad geographic region based on latitude"""
    if pd.isna(lat):
        return 'Unknown'
    elif lat > 50:
        return 'Far North'
    elif lat > 20:
        return 'Northern Temperate'
    elif lat > 0:
        return 'Northern Tropics'
    elif lat > -20:
        return 'Southern Tropics'
    elif lat > -50:
        return 'Southern Temperate'
    else:
        return 'Far South'

# Add region to both datasets
dplace_df['region'] = dplace_df['lat'].apply(assign_region)
dplace_filtered['region'] = dplace_filtered['lat'].apply(assign_region)

# Compare regional representation
print("\nSample distribution by region:")
print(f"  {'Region':<25} {'Full':>8} {'Filtered':>8} {'%Retained':>10}")
print("  " + "-" * 53)

regions = sorted(dplace_df['region'].unique())
total_full = 0
total_filtered = 0

for region in regions:
    count_full = (dplace_df['region'] == region).sum()
    count_filtered = (dplace_filtered['region'] == region).sum()
    pct_retained = 100 * count_filtered / count_full if count_full > 0 else 0
    print(f"  {region:<25} {count_full:>8} {count_filtered:>8} {pct_retained:>9.1f}%")
    total_full += count_full
    total_filtered += count_filtered

print("  " + "-" * 53)
print(f"  {'TOTAL':<25} {total_full:>8} {total_filtered:>8} {100*total_filtered/total_full:>9.1f}%")

In [ ]:
# Visualise geographic distribution - Filtered vs Full
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Full dataset
ax1 = axes[0]
scatter1 = ax1.scatter(dplace_df['lon'], dplace_df['lat'], alpha=0.5, s=30, c='lightcoral', edgecolors='none', label='Full dataset (2,087 cultures)')
ax1.set_xlim(-180, 180)
ax1.set_ylim(-90, 90)
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.set_title('Full D-PLACE Dataset\n(All 2,087 cultures, phylogenetically non-independent)')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='lower left')

# Plot 2: Phylo-filtered dataset
ax2 = axes[1]
scatter2 = ax2.scatter(dplace_filtered['lon'], dplace_filtered['lat'], alpha=0.7, s=40, c='steelblue', edgecolors='black', linewidth=0.5, label=f'Phylo-filtered ({len(dplace_filtered)} cultures)')
ax2.set_xlim(-180, 180)
ax2.set_ylim(-90, 90)
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.set_title('Phylogenetically-Filtered Dataset\n(One-per-language-family, ~150–200 cultures)')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='lower left')

plt.tight_layout()
plt.savefig(VIZ_PATH / 'geographic_distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Visualisation saved: geographic_distribution_comparison.png")

## Section 5: Summary Statistics & Quality Checks

In [ ]:
# Quality checks and summary statistics
print("\nQUALITY CHECKS - PHYLOGENETICALLY FILTERED DATASET:")
print("=" * 70)

# Check 1: No duplicate language families
lf_counts = dplace_filtered['language_family'].value_counts()
print(f"✓ Check 1: One culture per language family")
print(f"  All families have exactly 1 representative: {(lf_counts == 1).all()}")
print(f"  Number of language families: {len(lf_counts)}")

# Check 2: Geographic coverage
print(f"\n✓ Check 2: Geographic coverage")
print(f"  Non-null coordinates: {dplace_filtered[['lat', 'lon']].notna().any(axis=1).sum()} / {len(dplace_filtered)}")
print(f"  Latitude range: [{dplace_filtered['lat'].min():.1f}, {dplace_filtered['lat'].max():.1f}]")
print(f"  Longitude range: [{dplace_filtered['lon'].min():.1f}, {dplace_filtered['lon'].max():.1f}]")

# Check 3: Temporal coverage (ethnographic present)
print(f"\n✓ Check 3: Temporal coverage (D-PLACE ethnographic present: 1800–1950)")
time_cols = [col for col in dplace_filtered.columns if 'time' in col.lower()]
if 'time_start_ce' in dplace_filtered.columns:
    print(f"  Time range: {dplace_filtered['time_start_ce'].min():.0f}–{dplace_filtered['time_end_ce'].max():.0f}")
    print(f"  Non-null dates: {dplace_filtered[['time_start_ce', 'time_end_ce']].notna().all(axis=1).sum()} / {len(dplace_filtered)}")

# Check 4: Feature data completeness
numeric_cols = dplace_filtered.select_dtypes(include=[np.number]).columns
print(f"\n✓ Check 4: Feature data completeness")
print(f"  Total numeric columns: {len(numeric_cols)}")
missing_by_col = dplace_filtered[numeric_cols].isnull().sum()
complete_cols = (missing_by_col == 0).sum()
print(f"  Columns with no missing values: {complete_cols} / {len(numeric_cols)}")

# Summary metrics
print("\n" + "=" * 70)
print("PHYLOGENETIC FILTERING SUMMARY:")
print("=" * 70)
print(f"Full dataset (input):        {len(dplace_df):>6,} cultures from {dplace_df['language_family'].nunique():>4,} language families")
print(f"Filtered dataset (primary):  {len(dplace_filtered):>6,} cultures (one per family)")
print(f"Reduction ratio:             {len(dplace_filtered)/len(dplace_df):>6.1%}")
print(f"\nRobustness check dataset:    {len(dplace_full):>6,} cultures (full dataset, non-independent)")
print("\n✓ All checks passed!")

## Section 6: Save Datasets for Phase 4 Clustering

In [ ]:
# Save datasets for Phase 4 clustering
print("\nSAVING DATASETS FOR PHASE 4:")
print("=" * 70)

# Save phylogenetically-filtered dataset (PRIMARY ANALYSIS)
filtered_path = OUTPUTS_PATH / 'dplace_phylo_filtered.parquet'
dplace_filtered.to_parquet(filtered_path, index=False)
print(f"✓ Saved phylo-filtered dataset: {filtered_path.name}")
print(f"  Shape: {dplace_filtered.shape}")
print(f"  Size: {filtered_path.stat().st_size / 1_000_000:.1f} MB")

# Save full dataset (ROBUSTNESS CHECK)
full_path = OUTPUTS_PATH / 'dplace_full_for_robustness.parquet'
dplace_full.to_parquet(full_path, index=False)
print(f"\n✓ Saved full dataset (robustness): {full_path.name}")
print(f"  Shape: {dplace_full.shape}")
print(f"  Size: {full_path.stat().st_size / 1_000_000:.1f} MB")

# Create and save metadata summary
metadata = {
    'phylogenetic_filtering_applied': True,
    'filter_method': 'one_per_language_family',
    'random_state': 42,
    'date_created': pd.Timestamp.now().isoformat(),
    'datasets': {
        'phylo_filtered': {
            'n_cultures': len(dplace_filtered),
            'n_language_families': len(lf_counts),
            'use_case': 'Primary clustering analysis (phylogenetically independent)'
        },
        'full_dataset': {
            'n_cultures': len(dplace_full),
            'n_language_families': dplace_full['language_family'].nunique(),
            'use_case': 'Robustness check / sensitivity analysis'
        }
    },
    'reference': 'Mace & Pagel (1994). Current Anthropology 35(3):87–92'
}

import json
metadata_path = OUTPUTS_PATH / 'phylogenetic_filtering_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f"\n✓ Saved metadata: {metadata_path.name}")

print("\n" + "=" * 70)
print("✓ PHASE 08: Phylogenetic Filtering COMPLETE")
print("=" * 70)
print(f"\nNext step: Run notebook 09_clustering_pipeline.ipynb")
print(f"  - Load: {filtered_path.name}")
print(f"  - Perform: k-means and hierarchical clustering")
print(f"  - Output: cluster membership assignments and validation metrics")